In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def classify_error(rmse_normalized):
    """Classifica os erros baseados no RMSE normalizado."""
    if rmse_normalized < 0.1:
        return 'baixo'
    elif rmse_normalized < 0.3:
        return 'médio'
    else:
        return 'alto'

def calculate_metrics(file_path):
    # Carregar o arquivo CSV
    df = pd.read_csv(file_path)
    
    # Obter a coluna y_test
    y_test = df['y_test']
    
    # Inicializar um dicionário para salvar as métricas
    metrics = {"model": [], "mae": [], "rmse_normalized": [], "rmse_class": [], "r2": []}
    
    # Iterar sobre as colunas dos modelos
    for col in df.columns:
        if col.startswith("y_predict_"):  # Selecionar colunas preditas
            model_name = col.replace("y_predict_", "")
            y_pred = df[col]
            
            # Calcular MAE
            mae = mean_absolute_error(y_test, y_pred)
            
            # Calcular RMSE normalizado
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            rmse_normalized = rmse / np.mean(y_test)
            
            # Classificar o RMSE normalizado
            rmse_class = classify_error(rmse_normalized)
            
            # Calcular R²
            r2 = r2_score(y_test, y_pred)
            
            # Adicionar métricas ao dicionário
            metrics["model"].append(model_name)
            metrics["mae"].append(mae)
            metrics["rmse_normalized"].append(rmse_normalized)
            metrics["rmse_class"].append(rmse_class)
            metrics["r2"].append(r2)
    
    # Retornar como DataFrame
    return pd.DataFrame(metrics)

# Função para processar todos os arquivos em uma pasta
def process_all_files(folder_path, output_file):
    # Verificar arquivos na pasta
    files = [f for f in os.listdir(folder_path) if f.startswith('Vazao_bbr') and f.endswith('.csv')]
    
    if not files:
        print("Nenhum arquivo CSV encontrado na pasta.")
        return
    
    all_metrics = []
    
    # Iterar sobre os arquivos
    for file in files:
        file_path = os.path.join(folder_path, file)
        metrics = calculate_metrics(file_path)
        metrics["file"] = file  # Adicionar o nome do arquivo
        all_metrics.append(metrics)
    
    # Combinar todos os DataFrames de métricas
    combined_metrics = pd.concat(all_metrics, ignore_index=True)
    
    # Salvar no arquivo de saída
    combined_metrics.to_csv(output_file, index=False)
    print(f"Métricas consolidadas salvas em {output_file}")

# Caminho da pasta e arquivo de saída
output_file = "../../results/regression/teste.csv"
folder_path = '../../results/regression/predictions-regression-bysource'

# Processar arquivos e calcular métricas
process_all_files(folder_path, output_file)
